# Week 1, Day 5: Harness Engineering

## Introduction

This notebook hands UnderCover's claims queue to an agent that works on its own. Instead of writing the tools ourselves, we use the Claude Agent SDK, the harness behind Claude Code, which supplies the agent loop, file tools, a task list and subagents. We give it a folder (the "desk") holding five claims, the customer and policy records, the policy wording and the claims manual. It keeps a task list, hands each claim to a `claim-handler` subagent, and writes a handover note. We then score its decisions against the handlers' own.

Handler decisions are evaluation labels only. They are never copied into the desk, so the agent cannot see them.

## Learning goals

- Build a working folder that holds everything an agent needs to decide a claim.
- Configure a harness: instructions, a tool allow-list, a subagent and a spending cap.
- Watch an agent work by streaming its tool calls and messages.
- Score the agent's decisions against handler labels, and read what its misses mean.

## Before you begin

- Run this notebook from its saved location, so the relative path `../../../data` resolves to the repository's `data/` folder.
- Set `ANTHROPIC_API_KEY` in the repository's `.env` file. A run on five claims costs about 40 cents, and the run is capped at $1.00.
- Saved outputs can be stale. Rerun from the top for results that match the current code.

## Notebook map

1. **Setup**: imports and a key check.
2. **Part 1, The claims desk**: load the claims and labels, and build the desk folder.
3. **Part 2, Setting up the harness**: the instructions, the subagent, the options and a Windows adjustment.
4. **Part 3, Working the queue**: run the agent and stream what it does.
5. **Part 4, How it did**: score the decisions and read the handover note.
6. **Conclusion**: what the run shows and what it does not.

In [19]:
# Standard-library helpers for data files, environment variables, timing and the desk folder.
import json
import os
import shutil
import time
from pathlib import Path

# Tabular scoring in Part 4.
import pandas as pd
# The Claude Agent SDK is the harness: options, a subagent definition, and the message types it streams back.
from claude_agent_sdk import AgentDefinition, AssistantMessage, ClaudeAgentOptions, ResultMessage, TextBlock, ToolUseBlock, query
from dotenv import load_dotenv
# Render the agent's handover note as formatted Markdown.
from IPython.display import Markdown, display

In [2]:
# Allow .env to override an existing shell value; the SDK reads ANTHROPIC_API_KEY from the environment.
load_dotenv(override=True)

# Report only whether the key exists; never print the key itself.
if os.getenv('ANTHROPIC_API_KEY'):
    print('ANTHROPIC_API_KEY is set')
else:
    print('ANTHROPIC_API_KEY is not set')

ANTHROPIC_API_KEY is set


## Part 1: The claims desk

Load the claims queue and the handler labels, then build a folder that holds everything the agent needs. The agent works only inside this folder. Labels are used for scoring in Part 4 and are never written to the desk.

In [3]:
# This notebook lives three directories below the repository root.
DATA = Path("../../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding='utf-8'))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load('evals/held_out_claims.json' if held_out else 'claims.json')
    # Handler decisions are the answer key; they are used only to score the agent.
    decisions = load('evals/held_out_decisions.json' if held_out else 'evals/decisions.json')
    if tiny:
        chosen = {decision['claim_id'] for decision in decisions if decision['tiny']}
        claims = [claim for claim in claims if claim['claim_id'] in chosen]
        decisions = [decision for decision in decisions if decision['tiny']]
    labels = {decision['claim_id']: decision for decision in decisions}
    return claims, labels

In [4]:
# Use the five-claim subset: quick and cheap. Set tiny=False for the full queue (and a bigger bill).
claims, labels = get_sample_data(tiny=True)

# Preview the queue the agent will work.
for claim in claims:
    print(claim['claim_id'], claim['summary'])

CLM-1007 Phone dropped in the hotel shower
CLM-1015 Flew home from Paris on arrival after leaving a child at home
CLM-1017 Backpack snatched by a scooter rider in Naples
CLM-1020 Prescription sunglasses stolen by a monkey at Uluwatu Temple
CLM-1021 Tote bag snatched from her shoulder in Buenos Aires


In [5]:
DESK = Path('lab1_desk')


def make_desk(claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    # Start clean so decisions from an earlier run cannot be mistaken for this run's.
    shutil.rmtree(DESK, ignore_errors=True)
    for folder in ['claims', 'policies', 'customers', 'decisions']:
        (DESK / folder).mkdir(parents=True)
    # One file per claim, policy and customer, named by id so the agent can find them with Glob.
    for claim in claims:
        (DESK / 'claims' / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding='utf-8')
    for policy in load('policies.json'):
        (DESK / 'policies' / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding='utf-8')
    for customer in load('customers.json'):
        (DESK / 'customers' / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding='utf-8')
    # The rules the agent decides by, copied as they are.
    for name in ['policy_wording.md', 'claims_manual.md', 'company.json']:
        shutil.copy(DATA / name, DESK / name)


make_desk(claims)
# The decisions folder starts empty; the agent fills it.
sorted(path.name for path in DESK.iterdir())

['claims',
 'claims_manual.md',
 'company.json',
 'customers',
 'decisions',
 'policies',
 'policy_wording.md']

### Part 1 summary

The desk is a plain folder: one JSON file for each queued claim, policy and customer, the policy wording, the claims manual and the company limits, plus an empty `decisions/` folder. Rebuilding it on every run keeps results from different runs apart. The handler labels stay in the notebook's memory and never reach the desk.

## Part 2: Setting up the harness

Tell the agent who it is and what the folder holds, define a `claim-handler` subagent that decides one claim, and set the options that bound what the agent may do. A harness is mostly these choices: which instructions, which tools, how much autonomy and how much money.

In [ ]:
# The lead agent's instructions, appended to Claude Code's own system prompt. It plans and delegates; it does not decide.
INSTRUCTIONS = """
You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.
Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a task list with one task for each claim, and give each claim to the claim-handler agent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

# The subagent's instructions. It sees one claim at a time and writes the file that Part 4 scores.
HANDLER_INSTRUCTIONS = """
You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.
Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.
Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""

In [8]:
# Tools for the lead agent: file access, plus the task list and the Task tool that starts subagents.
TOOLS = ['Read',
         'Glob',
         'Grep',
         'Write',
         'Task',
         'TaskCreate',
         'TaskUpdate',
         'TaskList',
         'TaskGet']

# The subagent gets fewer tools than the lead: it can read and write files but cannot delegate further.
claim_handler = AgentDefinition(
    description='Decides one claim and writes its decision file. Give it the claim id.',
    prompt=HANDLER_INSTRUCTIONS,
    tools=['Read',
           'Glob',
           'Grep',
           'Write'])

options = ClaudeAgentOptions(
    model='claude-sonnet-5-5',
    effort='medium',
    # The agent's working directory is the desk, so its relative paths stay inside it.
    cwd=str(DESK),
    # Keep Claude Code's built-in prompt and append our instructions to it.
    system_prompt={
        'type': 'preset',
        'preset': 'claude_code',
        'append': INSTRUCTIONS
    },
    # tools makes these available; allowed_tools lets them run without asking.
    tools=TOOLS,
    allowed_tools=TOOLS,
    agents={'claim-handler': claim_handler},
    # Anything not allowed is denied instead of prompting, since nobody is at the keyboard.
    permission_mode='dontAsk',
    # Ignore settings files on this machine so the run behaves the same everywhere.
    setting_sources=[],
    env={'CLAUDE_CODE_DISABLE_AUTO_MEMORY': '1', 'ENABLE_CLAUDEAI_MCP_SERVERS': 'false'},
    # Stop the run if it spends more than this.
    max_budget_usd=1.00,
)

In [9]:
import sys

# On Windows the SDK needs an event loop that can start subprocesses, which Jupyter's loop may not support.
# This wrapper runs the SDK's query on its own loop in a thread and hands the messages back. Other systems skip it.
if sys.platform == 'win32':
    import asyncio
    import threading

    import claude_agent_sdk

    async def query(*, prompt, options):
        """The SDK's query, run on an event loop of its own in a thread, passing each message back here."""
        loop = asyncio.get_running_loop()
        messages = asyncio.Queue()

        async def run():
            try:
                async for message in claude_agent_sdk.query(prompt=prompt, options=options):
                    loop.call_soon_threadsafe(messages.put_nowait, message)
            except Exception as error:
                # Pass the error back so it is raised in the notebook instead of being lost in the thread.
                loop.call_soon_threadsafe(messages.put_nowait, error)
            # None marks the end of the stream, whether it finished or failed.
            loop.call_soon_threadsafe(messages.put_nowait, None)

        threading.Thread(target=asyncio.run,
                         args=(run(),), kwargs={'loop_factory': asyncio.ProactorEventLoop}).start()

        while (message := await messages.get()) is not None:
            if isinstance(message, Exception):
                raise message
            yield message

    print('Applied the Windows adjustment')
else:
    print('Not Windows, so nothing to do here.')

Not Windows, so nothing to do here.


### Part 2 summary

The harness has two roles. The lead agent plans: it keeps a task list and delegates each claim. The `claim-handler` subagent decides one claim at a time and writes its decision file. Safety comes from the options: an explicit tool list, `permission_mode='dontAsk'` so nothing unlisted can run, a working directory limited to the desk and a $1.00 budget cap. The Windows cell changes nothing on macOS or Linux.

## Part 3: Working the queue

Send one prompt and stream what the agent does. Each tool call prints as a short line, and messages from a subagent are indented so you can tell the lead's work from the handlers'.

In [10]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call.input
    # File tools carry a path; show only the file name to keep lines short.
    if 'file_path' in args:
        return f"{tool_call.name} {Path(args['file_path']).name}"
    # The subagent tool is called Task or Agent depending on the SDK version.
    if tool_call.name in ('Agent', 'Task'):
        return f"{tool_call.name}: {args.get('description')}"
    if tool_call.name == 'TaskCreate':
        return f"TaskCreate: {args.get('subject')}"
    if tool_call.name == 'TaskUpdate':
        return f"TaskUpdate: task {args.get('taskId')} {args.get('status', '')}"
    # Glob and Grep carry a pattern.
    if 'pattern' in args:
        return f"{tool_call.name} {args['pattern']}"
    return tool_call.name

In [20]:
result = None
start = time.time()

# Stream messages as the agent works, instead of waiting for the whole run to finish.
async for message in query(prompt='Please work the queue.', options=options):
    if isinstance(message, AssistantMessage):
        # parent_tool_use_id is set only on messages from a subagent, so use it to indent them.
        indent = "      " if message.parent_tool_use_id else ""
        for block in message.content:
            if isinstance(block, ToolUseBlock):
                print(indent + describe(block))
            elif isinstance(block, TextBlock) and block.text.strip():
                print(indent + block.text.strip())
    elif isinstance(message, ResultMessage):
        # The final message carries the totals for the run.
        result = message

print(f"\nFinished in {time.time() - start:.0f} seconds for ${result.total_cost_usd:.2f}")

Glob claims/*
Glob *
TaskCreate: Decide CLM-1007
TaskCreate: Decide CLM-1015
TaskCreate: Decide CLM-1017
TaskCreate: Decide CLM-1020
TaskCreate: Decide CLM-1021
Agent: Decide CLM-1007
Agent: Decide CLM-1015
Agent: Decide CLM-1017
Agent: Decide CLM-1020
Agent: Decide CLM-1021
      Read CLM-1017.json
      Glob **/CLM-1007*
      Glob **/CLM-1015*
      Read policy_wording.md
      Read policy_wording.md
      Read policy_wording.md
      Glob **/CLM-1020*
      Read claims_manual.md
      Read claims_manual.md
      Read claims_manual.md
      Read policy_wording.md
      Glob **/CLM-1021*
      Read claims_manual.md
      Read policy_wording.md
      Read claims_manual.md
All five claims are with the claim-handler agents. I'm waiting for them to finish before I write the handover.
      Read POL-2017.json
      Read CUS-3017.json
      Read CLM-1020.json
      Read CLM-1007.json
      Read CLM-1015.json
      Read CLM-1021.json
      Read POL-2020.json
      Read CUS-3020.json
      R

### Part 3 summary

In the saved run the lead agent created one task per claim, started five `claim-handler` subagents, waited for them, then wrote the handover. The indented lines are the subagents reading each claim, its policy and customer records, and the wording and manual before writing a decision. A run is non-deterministic, so tool order and cost will differ a little each time. If the run stops early at the budget cap, `result` may be missing decisions, and Part 4 will show them as `missing`.

## Part 4: How it did

Compare each decision file with the handler's label. A claim counts as right when the decision matches and the payout is within a dollar. Then read the handover note the agent wrote for the human handler.

In [21]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer.get('decision') == label['decision'] and abs(answer.get('payout', 0) - label['payout']) <= 1


rows = []
for claim_id, label in labels.items():
    path = DESK / 'decisions' / f"{claim_id}.json"
    # A claim the agent never decided is scored as wrong rather than skipped.
    answer = json.loads(path.read_text(encoding='utf-8')) if path.exists() else {'decision': 'missing', 'payout': 0}
    rows.append({
        'claim': claim_id,
        'decision': answer.get('decision'),
        'payout': answer.get('payout'),
        "handler's decision": label['decision'],
        "handler's payout": label['payout'],
        'right': is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results

4 of 5 right


,claim,decision,payout,handler's decision,handler's payout,right
0,CLM-1007,approve,50.0,approve,50.0,True
1,CLM-1015,decline,0.0,decline,0.0,True
2,CLM-1017,refer,0.0,refer,0.0,True
3,CLM-1020,approve,240.0,approve,240.0,True
4,CLM-1021,approve,1053.0,approve,825.0,False


In [22]:
# The agent's own summary for the human handler: what was decided and what needs attention.
display(Markdown((DESK / 'handover.md').read_text(encoding='utf-8')))

# Handover: claims queue

All 5 claims in the queue have a decision file in `decisions/`.

| Claim | Decision | Payout | Notes |
|---|---|---|---|
| CLM-1007 | Approve | $50 | Phone damage. $1,099 capped at the $150 Economy per-item limit (no Electronics Plus), less the $100 deductible. |
| CLM-1015 | Decline | $0 | Trip interruption because the customer left a child at home. Not one of the five covered reasons in section 2. |
| CLM-1017 | Refer | $0 | Theft in Naples. Police report is missing. |
| CLM-1020 | Approve | $240 | Sunglasses taken by a macaque. Treated as accidental loss, so no police report was needed. First plan, $0 deductible. |
| CLM-1021 | Approve | $1,053 | Snatch theft. $1,103 of items less the $50 Premium deductible. |

## Needs your attention

- **CLM-1021: sign-off needed.** The payout is over $1,000, so payment waits for your approval (manual s3).
- **CLM-1017: waiting on the customer.** We need the Carabinieri report (the Italian original is fine) within 30 days. Check that it was made within 24 hours of the theft. If it was, the claim looks payable at about $720 ($820 of items, with the rain jacket capped at $150, less the $100 deductible). I have not confirmed that the document request has gone out to the customer, so please check.
- **CLM-1015: expect a complaint.** The decline follows the wording strictly, and the customer may dispute it.


### Part 4 summary

In the saved run the agent matched the handlers on 4 of 5 claims. The miss is CLM-1021, a baggage theft with the Electronics Plus add-on. The agent approved $1,053, but the handler approved $825. Electronics Plus raises the per-item limit only for a closed list of devices that includes e-readers but not headphones or game consoles, so those two items stay at the $300 Premium limit. The agent treated every item the claim form marked "electronics" as covered. The decision was right and the payout was wrong, a reasoning error rather than a tooling one. The handover note did flag the claim for sign-off because it was over $1,000, so a human would have had a chance to catch it.

## Conclusion

This notebook gave an off-the-shelf harness a folder of claims and let it work the queue: it planned with a task list, delegated each claim to a subagent, wrote decision files and a handover, and was then scored against the handlers' labels. We wrote no tools. The work was in the desk layout, the two sets of instructions and the options that bound the agent.

In the saved run the agent got 4 of 5 claims right, and the one miss was a payout, not a decision: it did not apply the narrow scope of the Electronics Plus add-on. The harness did its job by flagging the high payout for human sign-off.

Five claims, one run and one model are a small sample. A single pass says little about reliability: rerun it to see how much it varies, check the full queue and the held-out claims, and read the subagents' rationales as well as the scores. Handler labels must stay out of the desk throughout. The usual fixes for a miss like this are better instructions, such as pointing the handler at add-on scopes in the wording, or a check on payouts before anything is paid.